# **Регуляризация: как меняются веса Ridge, Lasso и Elastic Net при росте $\lambda$**

### Туториал к уроку «Регуляризация: как выбрать одно решение из множества»

В уроке три штрафа разобраны на игрушечном примере с двумя совпадающими признаками: Ridge делит вклад между ними поровну и плавно сжимает его к нулю, Lasso может занулить вклад при конечном $\lambda$, Elastic Net зануляет, как Lasso, и делит, как Ridge. Здесь те же три модели обучены на California Housing, на тех же признаках и том же разбиении, что в тетради `HW1_linear_models_weights_and_predictions`, и показаны двумя интерактивными графиками Plotly:

1. веса всех восьми признаков в зависимости от $\lambda$, с переключателем Ridge / Lasso / Elastic Net;
2. веса трех моделей рядом при одном $\lambda$, который выбирается ползунком.

Заданий в тренажере по этой тетради нет.

## Целевые функции урока и параметры sklearn

В уроке все три задачи записаны без деления на число объектов $n$:

$$\text{Ridge:}\ \|y-X\beta\|_2^2+\lambda\|\beta\|_2^2,\qquad \text{Lasso:}\ \|y-X\beta\|_2^2+\lambda\|\beta\|_1,$$

$$\text{Elastic Net:}\ \|y-X\beta\|_2^2+\lambda_1\|\beta\|_1+\lambda_2\|\beta\|_2^2.$$

В sklearn `Ridge` записан так же, поэтому `alpha` $=\lambda$. `Lasso` и `ElasticNet` делят ошибку на $2n$:

$$\text{Lasso:}\ \frac{1}{2n}\|y-X\beta\|_2^2+\alpha\|\beta\|_1,\qquad \text{ElasticNet:}\ \frac{1}{2n}\|y-X\beta\|_2^2+\alpha\rho\|\beta\|_1+\frac{\alpha(1-\rho)}{2}\|\beta\|_2^2,$$

где $\rho$ — `l1_ratio`. Разделив целевую функцию урока на $2n$, получаем для Lasso `alpha` $=\lambda/(2n)$, а для Elastic Net — `alpha` $=\lambda_1/(2n)+\lambda_2/n$ и `l1_ratio` $=\dfrac{\lambda_1/(2n)}{\text{alpha}}$. Ниже везде $\lambda$ — в шкале урока, и у Elastic Net $\lambda_1=\lambda_2=\lambda$ (тогда `l1_ratio` $=1/3$ при любом $\lambda$). Свободный член sklearn не штрафует, как и в уроке «Регуляризация: как выбрать одно решение из множества».

## Данные

California Housing, разбиение и стандартизация — как в тетради `HW1_linear_models_weights_and_predictions`: `random_state=42`, `test_size=0.25`, `StandardScaler`, обученный на `X_train`.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet

RANDOM_STATE = 42

data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=RANDOM_STATE, test_size=0.25)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

labels = list(X.columns)
n = X_train_scaled.shape[0]
print('n =', n)

## Пути весов

Обучаем три модели на сетке из 51 значения $\lambda$ от $1$ до $10^5$ (равномерно в логарифмической шкале).

In [ ]:
lambdas = np.logspace(0, 5, 51)  # λ от 1 до 100000 в шкале урока

def fit_weights(lam):
    ridge = Ridge(alpha=lam).fit(X_train_scaled, y_train)
    lasso = Lasso(alpha=lam / (2 * n), max_iter=50000).fit(X_train_scaled, y_train)
    # Elastic Net с lambda_1 = lambda_2 = lam
    alpha_en = lam / (2 * n) + lam / n
    enet = ElasticNet(alpha=alpha_en, l1_ratio=(lam / (2 * n)) / alpha_en, max_iter=50000).fit(X_train_scaled, y_train)
    return ridge.coef_, lasso.coef_, enet.coef_

names = ['Ridge', 'Lasso', 'Elastic Net']
weights = {name: [] for name in names}
for lam in lambdas:
    for name, w in zip(names, fit_weights(lam)):
        weights[name].append(w)
paths = {name: pd.DataFrame(w, index=lambdas, columns=labels) for name, w in weights.items()}

# веса при самом маленьком и самом большом lambda сетки
pd.concat({name: p.iloc[[0, -1]].T for name, p in paths.items()}, axis=1).round(4)

### График 1. Веса признаков в зависимости от $\lambda$

Кнопки над графиком переключают модель. Ось $\lambda$ логарифмическая.

In [ ]:
colors = px.colors.qualitative.Plotly

fig = go.Figure()
for k, name in enumerate(names):
    for j, f in enumerate(labels):
        fig.add_trace(go.Scatter(x=lambdas, y=paths[name][f], mode='lines', name=f,
                                 line=dict(color=colors[j]), visible=(k == 0)))

buttons = [dict(label=name, method='update',
                args=[{'visible': [kk == k for kk in range(len(names)) for _ in labels]},
                      {'title': f'{name}: weights against λ'}])
           for k, name in enumerate(names)]
fig.update_layout(updatemenus=[dict(type='buttons', direction='right', buttons=buttons, x=0, y=1.18, xanchor='left')],
                  title='Ridge: weights against λ', height=520,
                  xaxis=dict(type='log', title='λ (lesson scale, log axis)'),
                  yaxis=dict(title='weight of a standardized feature'))
fig.show()

### График 2. Веса трех моделей при одном $\lambda$

Ползунок под графиком выбирает $\lambda$ из той же сетки; все три модели показаны рядом на одной оси.

In [ ]:
fig2 = make_subplots(rows=1, cols=3, subplot_titles=names, shared_yaxes=True)
for k, name in enumerate(names):
    fig2.add_trace(go.Bar(x=labels, y=paths[name].iloc[0].tolist(), marker_color=colors[:len(labels)], showlegend=False),
                   row=1, col=k + 1)

steps = [dict(method='restyle', label=f'{lam:.0f}',
              args=[{'y': [paths[name].iloc[i].tolist() for name in names]}])
         for i, lam in enumerate(lambdas)]
ymax = max(p.abs().max().max() for p in paths.values()) * 1.1
fig2.update_yaxes(range=[-ymax, ymax])
fig2.update_yaxes(title_text='weight of a standardized feature', row=1, col=1)
fig2.update_xaxes(tickangle=45)
fig2.update_layout(title='Weights of the three models at one λ (move the slider)', height=560,
                   sliders=[dict(active=0, currentvalue=dict(prefix='λ = '), steps=steps, pad=dict(t=90))])
fig2.show()

## С какого $\lambda$ вес становится нулем

Для каждого признака — наименьшее $\lambda$ сетки, начиная с которого вес равен нулю до конца сетки. `NaN` — вес не обнулился ни при одном $\lambda$ сетки.

In [ ]:
def first_zero(path):
    # наименьшее lambda сетки, начиная с которого вес признака равен нулю до конца сетки
    zero = (path.abs() < 1e-10).values
    tail = np.flip(np.cumprod(np.flip(zero, axis=0), axis=0), axis=0).astype(bool)
    return pd.Series([path.index[tail[:, j].argmax()] if tail[:, j].any() else np.nan for j in range(path.shape[1])],
                     index=path.columns)

zero_table = pd.DataFrame({name: first_zero(p) for name, p in paths.items()})
zero_table.sort_values('Lasso').round(0)

Веса Latitude и Longitude — пары признаков с корреляцией $-0.92$ на `X_train` — у трех моделей, каждое пятое значение сетки:

In [ ]:
lat_lon = pd.concat({name: p[['Latitude', 'Longitude']] for name, p in paths.items()}, axis=1)
lat_lon.iloc[::5].round(4)

## Что видно на графиках и в таблицах

- **Ridge** не обнуляет ни одного веса на всей сетке: все веса сжимаются плавно, и даже при $\lambda=10^5$ самый большой по модулю вес — у MedInc, $0.1057$. Сжатие не обязано сохранять знак: вес AveRooms при $\lambda=1$ равен $-0.3019$, а при $\lambda=10^5$ — $+0.0212$.
- **Lasso** убирает признаки по одному: Population с $\lambda\approx50$, AveRooms с $\lambda\approx794$, AveBedrms и AveOccup с $\lambda\approx1585$, Longitude с $\lambda\approx2512$, Latitude с $\lambda\approx3981$, HouseAge с $\lambda\approx6310$, MedInc последним, с $\lambda\approx25119$. Из пары Latitude и Longitude Lasso первым зануляет Longitude: при $\lambda\approx3162$ вес Latitude еще $-0.0084$, а Longitude уже $0$.
- **Elastic Net** зануляет веса, как Lasso, и почти в том же порядке; раньше уходят только Population ($\lambda\approx25$) и AveBedrms ($\lambda\approx1000$). До зануления он сжимает сильнее обеих моделей, потому что действуют оба штрафа сразу: при $\lambda=1000$ веса Latitude и Longitude у Elastic Net $-0.2850$ и $-0.2437$, у Ridge $-0.4950$ и $-0.4611$, у Lasso $-0.5120$ и $-0.4701$.

Связь с тетрадью `HW1_linear_models_weights_and_predictions`: там Lasso обучали с `alpha` из сетки sklearn, и `alpha=0.1` в шкале урока — это $\lambda=2n\cdot0.1=3096$. Это значение лежит между $\lambda\approx2512$, с которого Lasso зануляет Longitude, и $\lambda\approx3981$, с которого зануляет Latitude, — поэтому в ней при `alpha=0.1` нулевой вес у Longitude, а у Latitude еще нет.